# QuantBench Intraday SOXL v1

This notebook is the primary runtime evidence for the v1 QuantBench-inspired intraday data architecture. It validates the canonical SOXL/SOXX 1-minute schema, dual-source provider contract, daily-safe feature projection, and current live-data readiness.

External reference: arXiv `2504.18600` QuantBench-style benchmark framing: standardized data/task/evaluation surfaces before model claims.

## Governance

- Direction: `data`
- Role: `diagnose` / architecture intake, not a promoted training run
- Changed layer: `data` and candidate `feature` intake only
- Canonical baseline: existing daily SOXL/SOXX/VIX panel and qlib store
- Direct comparison baseline: prior Futu full-history daily K-line probe
- Hypothesis: SOXL minute bars should first land in a provider-normalized, replayable data surface, then become lagged daily features before any RL profile can use them.
- Acceptance criteria: canonical schema exists, Futu and Polygon provider adapters share one shape, registry can track intraday parquet metadata, and missing live provider state is explicit.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "soxl_quant").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CARD = ROOT / "runtime/data/card/quantbench_intraday_soxl_v1"
CANONICAL_PATH = ROOT / "data/processed/intraday/soxl_soxx_1m.parquet"
FEATURE_PATH = ROOT / "data/raw/external/intraday_features_soxl_1m.csv"

from src.soxl_quant.app.intraday_data import (
    build_canonical_intraday_panel,
    build_intraday_daily_features,
    normalize_intraday_bars,
)

CARD.mkdir(parents=True, exist_ok=True)

In [2]:
# Use existing canonical intraday cache when present; otherwise use a deterministic fixture
# so the notebook remains executable in offline CI and planning environments.
if CANONICAL_PATH.exists():
    canonical = pd.read_parquet(CANONICAL_PATH)
    source = "canonical_cache"
else:
    futu = normalize_intraday_bars(
        pd.DataFrame([
            {"datetime": "2026-06-04 09:30:00", "open": 10.0, "high": 10.5, "low": 9.9, "close": 10.2, "volume": 1000},
            {"datetime": "2026-06-04 15:59:00", "open": 10.2, "high": 10.8, "low": 10.1, "close": 10.7, "volume": 3000},
        ]),
        provider="futu", symbol="SOXL", interval="1m", timestamp_column="datetime"
    )
    polygon = normalize_intraday_bars(
        pd.DataFrame([
            {"datetime": "2026-06-04 09:30:00", "open": 10.0, "high": 10.5, "low": 9.9, "close": 10.21, "volume": 995},
            {"datetime": "2026-06-04 15:59:00", "open": 10.2, "high": 10.8, "low": 10.1, "close": 10.7, "volume": 3005},
        ]),
        provider="polygon", symbol="SOXL", interval="1m", timestamp_column="datetime"
    )
    canonical = build_canonical_intraday_panel([futu, polygon])
    source = "offline_fixture"

canonical.head()

,provider,symbol,interval,timestamp,date,session,open,high,low,close,volume,turnover,source_tz,adjusted,fetched_at,source_count,close_disagreement_abs
0,futu,SOXL,1m,2026-06-04 09:30:00-04:00,2026-06-04,regular,10.0,10.5,9.9,10.2,1000,0.0,America/New_York,True,2026-06-06T02:10:25.613415+00:00,2,0.01
1,futu,SOXL,1m,2026-06-04 15:59:00-04:00,2026-06-04,regular,10.2,10.8,10.1,10.7,3000,0.0,America/New_York,True,2026-06-06T02:10:25.613415+00:00,2,0.00


In [3]:
coverage = {
    "source": source,
    "rows": int(len(canonical)),
    "symbols": sorted(canonical["symbol"].dropna().unique().tolist()) if not canonical.empty else [],
    "providers": sorted(canonical["provider"].dropna().unique().tolist()) if not canonical.empty else [],
    "start": pd.to_datetime(canonical["timestamp"]).min().isoformat() if not canonical.empty else None,
    "end": pd.to_datetime(canonical["timestamp"]).max().isoformat() if not canonical.empty else None,
    "max_close_disagreement_abs": float(canonical.get("close_disagreement_abs", pd.Series([0.0])).max()) if not canonical.empty else 0.0,
}
coverage

{'source': 'offline_fixture',
 'rows': 2,
 'symbols': ['SOXL'],
 'providers': ['futu'],
 'start': '2026-06-04T09:30:00-04:00',
 'end': '2026-06-04T15:59:00-04:00',
 'max_close_disagreement_abs': 0.010000000000001563}

In [4]:
features = build_intraday_daily_features(canonical)
features

,date,INTRADAY_SOXL_BAR_COUNT_1M,INTRADAY_SOXL_OPEN_RANGE_RETURN_30M,INTRADAY_SOXL_CLOSE_LOCATION,INTRADAY_SOXL_REALIZED_VOL_1M,INTRADAY_SOXL_VOLUME_CONCENTRATION_CLOSE_30M
0,2026-06-05,2,0.07,0.888889,0.02451,0.75


In [5]:
decision = "architecture_ready_live_fetch_pending"
if source == "canonical_cache" and coverage["rows"] > 0:
    decision = "canonical_intraday_cache_available"

summary = {
    "experiment": "quantbench_intraday_soxl_v1",
    "decision": decision,
    "promotion_allowed": False,
    "training_allowed_from_this_card": False,
    "source": source,
    "coverage": coverage,
    "feature_rows": int(len(features)),
    "feature_columns": [col for col in features.columns if col != "date"],
    "live_fetch_command": "PYTHONPATH=. qlib_venv/bin/python scripts/fetch_intraday_data.py --providers futu,polygon --symbols SOXL,SOXX --interval 1m --start 2024-01-01 --end 2026-06-04 --session regular",
    "provider_requirements": {
        "futu": "OpenD reachable at 127.0.0.1:11111 with US quote permission",
        "polygon": "POLYGON_API_KEY set with stocks aggregate access",
    },
    "risk_notes": [
        "Minute features must be projected to daily rows with one-trading-day lag before training.",
        "Futu minute coverage must be measured empirically; do not assume full SOXL history.",
        "This card does not promote any actor or change production broker behavior.",
    ],
    "sidecars": ["summary.json"],
}
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
summary

{'experiment': 'quantbench_intraday_soxl_v1',
 'decision': 'architecture_ready_live_fetch_pending',
 'promotion_allowed': False,
 'training_allowed_from_this_card': False,
 'source': 'offline_fixture',
 'coverage': {'source': 'offline_fixture',
  'rows': 2,
  'symbols': ['SOXL'],
  'providers': ['futu'],
  'start': '2026-06-04T09:30:00-04:00',
  'end': '2026-06-04T15:59:00-04:00',
  'max_close_disagreement_abs': 0.010000000000001563},
 'feature_rows': 1,
 'feature_columns': ['INTRADAY_SOXL_BAR_COUNT_1M',
  'INTRADAY_SOXL_OPEN_RANGE_RETURN_30M',
  'INTRADAY_SOXL_CLOSE_LOCATION',
  'INTRADAY_SOXL_REALIZED_VOL_1M',
  'INTRADAY_SOXL_VOLUME_CONCENTRATION_CLOSE_30M'],
 'live_fetch_command': 'PYTHONPATH=. qlib_venv/bin/python scripts/fetch_intraday_data.py --providers futu,polygon --symbols SOXL,SOXX --interval 1m --start 2024-01-01 --end 2026-06-04 --session regular',
 'provider_requirements': {'futu': 'OpenD reachable at 127.0.0.1:11111 with US quote permission',
  'polygon': 'POLYGON_API_K

## Conclusion

The v1 intraday architecture is ready as a repository interface: provider adapters normalize Futu and Polygon into one canonical schema, registry support can track intraday parquet assets, and the first daily-safe feature projection is explicitly lagged. This notebook used the canonical cache if present, otherwise an offline fixture, so it does not claim live SOXL minute coverage yet. The next action is to run the recorded live fetch command on a machine with OpenD and `POLYGON_API_KEY`, then rerun this notebook to replace fixture evidence with real provider coverage.